<div style="background: linear-gradient(135deg, #0d1117 0%, #161b22 50%, #0d1117 100%); border: 2px solid #30363d; border-radius: 12px; padding: 30px; margin-bottom: 20px;">

<h1 style="color: #f0f6fc; text-align: center; font-size: 2em; margin: 0 0 12px 0;">
  🔐 블록 암호화 실험 — ResNet50 기반 폐암 분류
</h1>
<p style="color: #8b949e; text-align: center; font-size: 1.05em; margin: 0 0 20px 0;">
  Block Encryption Experiment · Privacy-Preserving Lung Cancer Classification
</p>

<hr style="border: none; border-top: 1px solid #30363d; margin: 20px 0;" />

<table style="width: 100%; border-collapse: collapse; color: #c9d1d9; font-size: 0.95em;">
  <tr>
    <td style="padding: 6px 16px;">🖼️ <b>이미지 크기</b></td><td style="padding: 6px;">512 × 512 고정</td>
    <td style="padding: 6px 16px;">🧩 <b>블록 크기</b></td><td style="padding: 6px;">512 → 256 → 128 → 64 → 32 → 16 → 8 → 4 → 2 → 1</td>
  </tr>
  <tr style="background: #161b22;">
    <td style="padding: 6px 16px;">🏗️ <b>모델</b></td><td style="padding: 6px;">ResNet50 (ImageNet pretrained)</td>
    <td style="padding: 6px 16px;">📦 <b>오버샘플링</b></td><td style="padding: 6px;">클래스별 1,000장</td>
  </tr>
  <tr>
    <td style="padding: 6px 16px;">🔀 <b>K-Fold</b></td><td style="padding: 6px;">7-Fold Stratified CV</td>
    <td style="padding: 6px 16px;">🎯 <b>TTA</b></td><td style="padding: 6px;">20회 앙상블</td>
  </tr>
  <tr style="background: #161b22;">
    <td style="padding: 6px 16px;">📊 <b>시각화</b></td><td style="padding: 6px;">그래프 10개 + 과적합/과소적합 진단 + GPU 모니터</td>
    <td style="padding: 6px 16px;">💾 <b>원본 보존</b></td><td style="padding: 6px;">✅ BASE_TRAIN 절대 수정 없음</td>
  </tr>
</table>
</div>

---

## 📋 실험 목차

| # | 셀 | 내용 |
|---|---|---|
| 1 | Cell 1 | 라이브러리 임포트 |
| 2 | Cell 2 | 한글 폰트 설정 |
| 3 | **Cell 3** | **GPU 설정 & 모니터링 대시보드** ← 🆕 |
| 4 | Cell 4 | 실험 설정 (하이퍼파라미터) |
| 5 | Cell 5 | 블록 암호화 함수 |
| 6 | Cell 6 | 오버샘플링 |
| 7 | Cell 7 | Train/Test 분리 |
| 8 | Cell 8 | tf.data 파이프라인 |
| 9 | Cell 9 | 학습 기법 (Focal Loss / Cosine LR / SWA) |
| 10 | Cell 10 | 모델 빌드 (ResNet50) |
| 11 | **Cell 10b** | **학습 히스토리 기록 & 과적합/과소적합 진단** ← 🆕 |
| 12 | Cell 11 | 단일 블록 실험 함수 |
| 13 | Cell 12 | 전체 실험 루프 |
| 14 | Cell 13 | 결과 표 출력 |
| 15 | **Cell 13b** | **과적합/과소적합 상태 자동 진단 표** ← 🆕 |
| 16 | Cell 14 | 그래프 시각화 (10개) |


## 1️⃣ 라이브러리 임포트

In [ ]:
# Cell 1: 라이브러리 임포트
import os, math, shutil, warnings
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
from sklearn.metrics import confusion_matrix, roc_auc_score
from sklearn.utils.class_weight import compute_class_weight
from sklearn.model_selection import StratifiedKFold
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.layers import (Dense, GlobalAveragePooling2D, GlobalMaxPooling2D,
                                      Dropout, BatchNormalization, Concatenate)
from tensorflow.keras.models import Model, load_model
from tensorflow.keras.optimizers import AdamW
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, Callback
from tensorflow.keras.regularizers import l2
from tensorflow.keras import mixed_precision
import tensorflow.keras.backend as K
from PIL import Image
import random, json as json_module, gc, subprocess, time

warnings.filterwarnings("ignore")

print("✅ 라이브러리 임포트 완료")
print(f"   TensorFlow: {tf.__version__}")
print(f"   Python 환경 준비 완료")


## 2️⃣ 한글 폰트 설정

In [ ]:
# Cell 2: 한글 폰트 설정 (matplotlib 한글·기호 완전 지원)
import matplotlib
import matplotlib.font_manager as fm
import glob

font_path = "/usr/share/fonts/truetype/nanum/NanumGothic.ttf"

# 폰트 캐시 초기화
cache_dir = matplotlib.get_cachedir()
for f in glob.glob(os.path.join(cache_dir, "*.json")) + glob.glob(os.path.join(cache_dir, "*.cache")):
    try: os.remove(f)
    except: pass

fm.fontManager.addfont(font_path)
font_name = fm.FontProperties(fname=font_path).get_name()
matplotlib.rcParams["font.family"]       = "sans-serif"
matplotlib.rcParams["font.sans-serif"]   = [font_name, "DejaVu Sans"]
matplotlib.rcParams["axes.unicode_minus"] = False

# 공통 스타일 적용
matplotlib.rcParams.update({
    "figure.facecolor":  "#0d1117",
    "axes.facecolor":    "#161b22",
    "axes.edgecolor":    "#444",
    "axes.labelcolor":   "#aaa",
    "xtick.color":       "#aaa",
    "ytick.color":       "#aaa",
    "grid.color":        "#333",
    "grid.linestyle":    "--",
    "grid.alpha":        0.5,
    "legend.facecolor":  "#222",
    "legend.labelcolor": "white",
    "text.color":        "white",
})

print(f"✅ 한글 폰트 적용: {font_name}")
print(f"✅ 다크테마 rcParams 전역 적용 완료")


## 3️⃣ GPU 설정 & 실시간 모니터링 🆕

> GPU가 제대로 감지되고 있는지, 메모리는 충분한지 한눈에 확인합니다.


In [ ]:
# Cell 3: GPU 설정 + 모니터링 대시보드
# ──────────────────────────────────────────────────────
# Mixed precision + JIT 설정
mixed_precision.set_global_policy("mixed_float16")
tf.config.optimizer.set_jit(False)

gpus = tf.config.list_physical_devices("GPU")
print("=" * 55)
print("🖥️  GPU 환경 점검")
print("=" * 55)

if not gpus:
    print("⚠️  GPU 미감지 — CPU 모드로 실행됩니다 (속도 느림)")
else:
    for i, gpu in enumerate(gpus):
        print(f"  ✅ GPU {i}: {gpu.name}")
    try:
        result = subprocess.run(
            ["nvidia-smi",
             "--query-gpu=name,memory.total,memory.free,memory.used,utilization.gpu,temperature.gpu",
             "--format=csv,noheader,nounits"],
            capture_output=True, text=True, timeout=10
        )
        if result.returncode == 0:
            for line in result.stdout.strip().split("\n"):
                parts = [p.strip() for p in line.split(",")]
                if len(parts) >= 6:
                    name, total, free, used, util, temp = parts[:6]
                    used_pct = round(int(used) / int(total) * 100, 1)
                    free_pct = 100 - used_pct
                    bar_used = "█" * int(used_pct / 5)
                    bar_free = "░" * (20 - int(used_pct / 5))
                    print(f"\n  GPU 이름   : {name}")
                    print(f"  VRAM 총량  : {int(total):,} MiB")
                    print(f"  사용 중    : {int(used):,} MiB ({used_pct}%)")
                    print(f"  여유       : {int(free):,} MiB ({free_pct}%)")
                    print(f"  VRAM 사용률: [{bar_used}{bar_free}] {used_pct}%")
                    print(f"  GPU 활용률 : {util}%")
                    print(f"  온도       : {temp}°C")

                    # 상태 판단
                    print()
                    if int(free) < 2000:
                        print("  🔴 경고: VRAM 여유 2GB 미만 → OOM 위험! BATCH_SIZE를 줄이세요.")
                    elif int(free) < 4000:
                        print("  🟡 주의: VRAM 여유 4GB 미만 → 배치 크기 모니터링 권장")
                    else:
                        print("  🟢 정상: VRAM 충분 — 학습 준비 완료")
    except Exception as e:
        print(f"  nvidia-smi 조회 실패: {e}")

print("=" * 55)
print(f"  Mixed Precision: float16 ✅")
print(f"  XLA JIT: 비활성화 ✅")
print("=" * 55)

# ── GPU 모니터링 함수 (학습 중 호출 가능) ──────────────────────────────────
def gpu_status(label=""):
    """학습 중 GPU 사용률/온도를 간단히 출력하는 유틸 함수."""
    try:
        r = subprocess.run(
            ["nvidia-smi",
             "--query-gpu=memory.used,memory.total,utilization.gpu,temperature.gpu",
             "--format=csv,noheader,nounits"],
            capture_output=True, text=True, timeout=5
        )
        if r.returncode == 0:
            parts = [p.strip() for p in r.stdout.strip().split(",")]
            used, total, util, temp = int(parts[0]), int(parts[1]), parts[2], parts[3]
            pct = round(used / total * 100, 1)
            tag = f"[{label}] " if label else ""
            print(f"  {tag}GPU: {used}/{total}MiB ({pct}%) | 활용률: {util}% | 온도: {temp}°C")
    except:
        pass

# ── GPU 모니터링 콜백 ───────────────────────────────────────────────────────
class GPUMonitorCallback(Callback):
    """매 N 에포크마다 GPU 상태를 출력하는 Keras 콜백."""
    def __init__(self, every_n_epochs=5):
        super().__init__()
        self.every_n = every_n_epochs

    def on_epoch_end(self, epoch, logs=None):
        if (epoch + 1) % self.every_n == 0:
            gpu_status(f"Epoch {epoch+1}")

print("\n✅ gpu_status() / GPUMonitorCallback 정의 완료")
print("   → 학습 중 gpu_status('태그') 로 언제든 GPU 확인 가능")


## 4️⃣ 실험 설정 (하이퍼파라미터)

In [ ]:
# Cell 4: 실험 설정
# 블러 노트북과 동일한 순서/시드/분할 규칙 사용
BLOCK_SIZES  = [512, 256, 128, 64, 32, 16, 8, 4, 2, 1]

BASE_TRAIN   = "/workspace/Train"       # 원본 학습 데이터 (절대 수정하지 않음)
WORKSPACE    = "/workspace"
IMG_SIZE     = 512          # 이미지 512×512 고정
BATCH_SIZE   = 4
EPOCHS_S1    = 30
EPOCHS_S2    = 60
NUM_CLASSES  = 3
SEED         = 42
N_FOLDS      = 7
TTA_STEPS    = 20
SWA_START    = 20
TARGET_COUNT = 1000         # 각 클래스별 1000장으로 오버샘플링
TEST_RATIO   = 0.10
AUTOTUNE     = tf.data.AUTOTUNE

# 같은 설정으로 새 결과/분할을 만들기 위한 버전 태그
EXPERIMENT_VERSION = "same_settings_v1"
ORIGINAL_SPLIT_DIR = os.path.join(
    WORKSPACE, f"AllExp_Split_ORIGINAL_seed{SEED}_{EXPERIMENT_VERSION}"
)
IMAGE_EXTENSIONS = (".jpg", ".jpeg", ".png", ".bmp")

# 학습 히스토리 저장소 (과적합/과소적합 진단용)
ALL_HISTORIES = {}   # { block_size: { fold: history_dict } }

def set_reproducible(seed=SEED):
    os.environ["PYTHONHASHSEED"] = str(seed)
    os.environ["TF_DETERMINISTIC_OPS"] = "1"
    random.seed(seed)
    np.random.seed(seed)
    try:
        tf.keras.utils.set_random_seed(seed)
    except AttributeError:
        tf.random.set_seed(seed)
    try:
        tf.config.experimental.enable_op_determinism()
    except Exception:
        pass

def list_class_dirs(directory):
    return sorted([d for d in os.listdir(directory)
                   if os.path.isdir(os.path.join(directory, d))])

def list_image_files(directory):
    if not os.path.exists(directory):
        return []
    return sorted([f for f in os.listdir(directory)
                   if f.lower().endswith(IMAGE_EXTENSIONS)])

set_reproducible(SEED)

print("┌" + "─" * 62 + "┐")
print("│           ✅ 실험 설정 완료                                  │")
print("├" + "─" * 62 + "┤")
print(f"│  이미지 크기     : {IMG_SIZE}×{IMG_SIZE} (고정){' '*31}│")
print(f"│  블록 크기      : {str(BLOCK_SIZES)[:42]:<42}│")
print(f"│  클래스별 목표   : {TARGET_COUNT}장{' '*42}│")
print(f"│  K-Fold         : {N_FOLDS}{' '*45}│")
print(f"│  TTA 횟수       : {TTA_STEPS}회{' '*43}│")
print(f"│  배치 크기      : {BATCH_SIZE}{' '*46}│")
print(f"│  공통 원본 split: {os.path.basename(ORIGINAL_SPLIT_DIR)[:42]:<42}│")
print(f"│  실험 버전      : {EXPERIMENT_VERSION:<42}│")
print("└" + "─" * 62 + "┘")


## 5️⃣ 블록 암호화 함수

In [ ]:
# Cell 5: 블록 암호화 함수
def block_scramble(img_array, seed, block_size):
    h, w, c = img_array.shape
    if block_size >= min(h, w):
        return img_array.copy()

    pad_h = (block_size - h % block_size) % block_size
    pad_w = (block_size - w % block_size) % block_size
    if pad_h > 0 or pad_w > 0:
        img_array = np.pad(img_array, ((0, pad_h), (0, pad_w), (0, 0)), mode="reflect")

    ph, pw   = img_array.shape[:2]
    n_h, n_w = ph // block_size, pw // block_size
    blocks   = [img_array[i*block_size:(i+1)*block_size, j*block_size:(j+1)*block_size]
                for i in range(n_h) for j in range(n_w)]

    rng      = np.random.default_rng(seed)
    order    = rng.permutation(n_h * n_w)
    result   = np.zeros_like(img_array)

    for new_idx, old_idx in enumerate(order):
        i, j = new_idx // n_w, new_idx % n_w
        result[i*block_size:(i+1)*block_size, j*block_size:(j+1)*block_size] = blocks[old_idx]

    return result[:h, :w]


def generate_block_dataset(block_size, seed=SEED):
    """
    block_size=512 → 원본(BASE_TRAIN) 그대로 사용 (파일 복사 없음)
    block_size<512 → Train_Block{block_size}/ 에 암호화 이미지 저장
    BASE_TRAIN 원본 파일은 절대 수정하지 않음.
    """
    if block_size >= IMG_SIZE:
        print(f"  Block{block_size}: 원본 사용 (원본 건드리지 않음)")
        return BASE_TRAIN

    out_dir = os.path.join(WORKSPACE, f"Train_Block{block_size}")
    classes = list_class_dirs(BASE_TRAIN)

    total_src = sum(len(list_image_files(os.path.join(BASE_TRAIN, c))) for c in classes)
    total_ex = (
        sum(len(list_image_files(os.path.join(out_dir, c)))
            for c in classes if os.path.exists(os.path.join(out_dir, c)))
        if os.path.exists(out_dir) else 0
    )

    if total_ex >= total_src:
        print(f"  Block{block_size}: 이미 존재 ({total_ex}장) → 스킵")
        return out_dir

    print(f"  Block{block_size}: 블록 암호화 처리 중 ({total_src}장)...")
    for cls in classes:
        os.makedirs(os.path.join(out_dir, cls), exist_ok=True)
        for f in list_image_files(os.path.join(BASE_TRAIN, cls)):
            dst = os.path.join(out_dir, cls, f)
            if os.path.exists(dst):
                continue
            try:
                img = Image.open(os.path.join(BASE_TRAIN, cls, f)).convert("RGB")
                img_array = np.array(img, dtype=np.uint8)
                Image.fromarray(block_scramble(img_array, seed, block_size)).save(
                    dst, "JPEG", quality=95
                )
            except Exception:
                pass
    print(f"  Block{block_size}: 완료! (원본 파일 보존됨)")
    return out_dir


## 6️⃣ 오버샘플링 (클래스별 1,000장)

In [ ]:
# Cell 6: 오버샘플링 (각 클래스 1000장, 원본 파일 보존)
def oversample_dir(base_dir, target_count=1000, seed=SEED):
    """원본(base_dir) 파일을 건드리지 않고, 버전이 붙은 oversampled 폴더에 복사+증강."""
    out_dir = base_dir.rstrip("/") + f"_oversampled_{EXPERIMENT_VERSION}_seed{seed}"
    classes = list_class_dirs(base_dir)

    if os.path.exists(out_dir):
        all_done = all(
            len(list_image_files(os.path.join(out_dir, c))) >= target_count
            for c in classes if os.path.exists(os.path.join(out_dir, c))
        ) and all(os.path.exists(os.path.join(out_dir, c)) for c in classes)
        if all_done:
            total = sum(len(list_image_files(os.path.join(out_dir, c))) for c in classes)
            print(f"    오버샘플링 이미 완료 ({total}장) → 재사용")
            return out_dir

    rng = random.Random(seed)

    for cls in classes:
        os.makedirs(os.path.join(out_dir, cls), exist_ok=True)
        src_files = [f for f in list_image_files(os.path.join(base_dir, cls))
                     if not f.startswith("aug_")]
        for f in src_files:
            dst = os.path.join(out_dir, cls, f)
            if not os.path.exists(dst):
                shutil.copy2(os.path.join(base_dir, cls, f), dst)

    for cls in classes:
        cls_out = os.path.join(out_dir, cls)
        files = list_image_files(cls_out)
        src_files = [f for f in list_image_files(os.path.join(base_dir, cls))
                     if not f.startswith("aug_")]
        needed = target_count - len(files)
        if needed <= 0:
            print(f"    {cls}: {len(files)}장 (목표 {target_count}장 달성)")
            continue
        if not src_files:
            print(f"    {cls}: 원본 이미지가 없어 증강을 건너뜀")
            continue
        print(f"    {cls}: {len(files)}장 → {target_count}장 ({needed}장 증강 생성)")
        generated = 0
        while generated < needed:
            src_f = rng.choice(src_files)
            try:
                img = Image.open(os.path.join(base_dir, cls, src_f)).convert("RGB")
                img = img.rotate(rng.uniform(-30, 30), expand=False)
                if rng.random() > 0.5: img = img.transpose(Image.FLIP_LEFT_RIGHT)
                if rng.random() > 0.5: img = img.transpose(Image.FLIP_TOP_BOTTOM)
                from PIL import ImageEnhance
                img = ImageEnhance.Brightness(img).enhance(rng.uniform(0.7, 1.3))
                img.save(os.path.join(cls_out, f"aug_{generated:05d}_{src_f}"),
                         "JPEG", quality=95)
                generated += 1
            except Exception:
                continue

    total = sum(len(list_image_files(os.path.join(out_dir, c))) for c in classes)
    print(f"    ✅ 오버샘플링 완료: {total}장 → {out_dir}")
    print(f"    ✅ 원본 파일({base_dir}) 보존 확인")
    return out_dir


## 7️⃣ Train / Test 분리

In [ ]:
# Cell 7: Train/Test 분리
def prepare_split(base_dir, split_dir, test_ratio=0.10, seed=SEED):
    test_dir  = os.path.join(split_dir, "test")
    kfold_dir = os.path.join(split_dir, "kfold_data")
    if os.path.exists(test_dir) and os.path.exists(kfold_dir):
        print(f"    Split 폴더 이미 존재 → 재사용: {split_dir}")
        return

    rng = np.random.default_rng(seed)
    classes = list_class_dirs(base_dir)
    for cls in classes:
        os.makedirs(os.path.join(test_dir,  cls), exist_ok=True)
        os.makedirs(os.path.join(kfold_dir, cls), exist_ok=True)
    for cls in classes:
        files = list_image_files(os.path.join(base_dir, cls))
        rng.shuffle(files)
        n_test = int(len(files) * test_ratio)
        for f in files[:n_test]:
            shutil.copy2(os.path.join(base_dir, cls, f), os.path.join(test_dir, cls, f))
        for f in files[n_test:]:
            shutil.copy2(os.path.join(base_dir, cls, f), os.path.join(kfold_dir, cls, f))
        print(f"    {cls}: kfold={len(files)-n_test}, test={n_test}")


## 8️⃣ tf.data 파이프라인 (512×512 고정)

In [ ]:
# Cell 8: tf.data 파이프라인 (512×512 고정)
DATA_OPTIONS = tf.data.Options()
DATA_OPTIONS.experimental_deterministic = True

def _deterministic(ds):
    return ds.with_options(DATA_OPTIONS)

def get_paths_labels(directory, class_names):
    paths, labels = [], []
    for idx, cls in enumerate(class_names):
        cls_dir = os.path.join(directory, cls)
        if not os.path.exists(cls_dir): continue
        for f in list_image_files(cls_dir):
            paths.append(os.path.join(cls_dir, f))
            one_hot = [0.0] * len(class_names)
            one_hot[idx] = 1.0
            labels.append(one_hot)
    return paths, labels

@tf.function
def load_image(path, label):
    raw = tf.io.read_file(path)
    img = tf.image.decode_jpeg(raw, channels=3)
    img = tf.image.resize(img, [IMG_SIZE, IMG_SIZE])
    return tf.cast(img, tf.float32), label

@tf.function
def augment_train(img, label):
    img = tf.cast(img, tf.float32)
    img = tf.image.random_flip_left_right(img)
    img = tf.image.random_flip_up_down(img)
    img = tf.image.random_brightness(img, max_delta=0.25 * 255)
    img = tf.image.random_contrast(img, lower=0.75, upper=1.25)
    noise = tf.random.normal(shape=tf.shape(img), mean=0.0, stddev=5.0, dtype=tf.float32)
    img = tf.clip_by_value(img + noise, 0.0, 255.0)
    return img, label

@tf.function
def mixup_batch(x, y, alpha=0.3):
    batch_size = tf.shape(x)[0]
    idx = tf.random.shuffle(tf.range(batch_size))
    lam = tf.cast(tf.random.uniform([], minval=alpha, maxval=1.0 - alpha + 0.001), tf.float32)
    x = tf.cast(x, tf.float32); y = tf.cast(y, tf.float32)
    return lam * x + (1 - lam) * tf.gather(x, idx), lam * y + (1 - lam) * tf.gather(y, idx)

@tf.function
def cutmix_batch(x, y, alpha=0.3):
    batch_size = tf.shape(x)[0]
    idx   = tf.random.shuffle(tf.range(batch_size))
    lam   = tf.cast(tf.random.uniform([], alpha, 1.0 - alpha), tf.float32)
    h     = tf.cast(tf.shape(x)[1], tf.float32)
    w     = tf.cast(tf.shape(x)[2], tf.float32)
    cut_h = tf.cast(h * tf.math.sqrt(1.0 - lam), tf.int32)
    cut_w = tf.cast(w * tf.math.sqrt(1.0 - lam), tf.int32)
    cx    = tf.random.uniform([], 0, tf.cast(h, tf.int32), dtype=tf.int32)
    cy    = tf.random.uniform([], 0, tf.cast(w, tf.int32), dtype=tf.int32)
    x1    = tf.clip_by_value(cx - cut_h // 2, 0, tf.cast(h, tf.int32))
    y1    = tf.clip_by_value(cy - cut_w // 2, 0, tf.cast(w, tf.int32))
    x2    = tf.clip_by_value(cx + cut_h // 2, 0, tf.cast(h, tf.int32))
    y2    = tf.clip_by_value(cy + cut_w // 2, 0, tf.cast(w, tf.int32))
    mask  = tf.ones([x2 - x1, y2 - y1, tf.shape(x)[3]])
    paddings = [[x1, tf.shape(x)[1] - x2], [y1, tf.shape(x)[2] - y2], [0, 0]]
    mask  = tf.pad(mask, paddings)
    mask  = tf.expand_dims(mask, 0)
    x     = tf.cast(x, tf.float32)
    mixed = x * (1.0 - mask) + tf.gather(x, idx) * mask
    lam_actual = 1.0 - tf.cast((x2 - x1) * (y2 - y1), tf.float32) / (h * w)
    y     = tf.cast(y, tf.float32)
    mixed_y = lam_actual * y + (1.0 - lam_actual) * tf.gather(y, idx)
    return mixed, mixed_y

def apply_mixup(ds, alpha=0.3):
    return _deterministic(ds.map(lambda x, y: mixup_batch(x, y, alpha),
                                 num_parallel_calls=AUTOTUNE)).prefetch(1)

def apply_cutmix(ds, alpha=0.3):
    return _deterministic(ds.map(lambda x, y: cutmix_batch(x, y, alpha),
                                 num_parallel_calls=AUTOTUNE)).prefetch(1)

def apply_mixup_cutmix(ds, alpha=0.3, cutmix_prob=0.5):
    def _apply(x, y):
        use_cutmix = tf.random.uniform([]) < cutmix_prob
        mx, my = mixup_batch(x, y, alpha)
        cx, cy = cutmix_batch(x, y, alpha)
        return tf.cond(use_cutmix, lambda: (cx, cy), lambda: (mx, my))
    return _deterministic(ds.map(_apply, num_parallel_calls=AUTOTUNE)).prefetch(1)

@tf.function
def augment_tta(img, label):
    img = tf.cast(img, tf.float32)
    img = tf.image.random_flip_left_right(img)
    img = tf.image.random_brightness(img, max_delta=0.15 * 255)
    img = tf.image.random_contrast(img, lower=0.85, upper=1.15)
    return img, label

def make_train_ds(directory, class_names, batch_size=BATCH_SIZE):
    paths, labels = get_paths_labels(directory, class_names)
    n  = len(paths)
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    ds = _deterministic(ds.shuffle(min(n, 1000), seed=SEED, reshuffle_each_iteration=True))
    ds = _deterministic(ds.map(load_image, num_parallel_calls=AUTOTUNE))
    ds = _deterministic(ds.map(augment_train, num_parallel_calls=AUTOTUNE))
    ds = ds.batch(batch_size, drop_remainder=True)
    ds = ds.prefetch(2)
    return ds, n

def make_eval_ds(directory, class_names, batch_size=BATCH_SIZE):
    paths, labels = get_paths_labels(directory, class_names)
    n  = len(paths)
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    ds = _deterministic(ds.map(load_image, num_parallel_calls=AUTOTUNE))
    ds = ds.batch(batch_size, drop_remainder=False)
    ds = ds.prefetch(1)
    return ds, n, paths

def make_tta_ds(directory, class_names, batch_size=BATCH_SIZE):
    paths, labels = get_paths_labels(directory, class_names)
    n  = len(paths)
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    ds = _deterministic(ds.map(load_image, num_parallel_calls=AUTOTUNE))
    ds = ds.batch(batch_size, drop_remainder=False)
    ds = _deterministic(ds.map(lambda x, y: augment_tta(x, y), num_parallel_calls=AUTOTUNE))
    ds = ds.prefetch(1)
    return ds, n

print(f"✅ tf.data 파이프라인 정의 완료 (이미지 {IMG_SIZE}×{IMG_SIZE} 고정, deterministic=True)")


## 9️⃣ 학습 기법 (Focal Loss / Cosine LR / SWA)

In [ ]:
# Cell 9: 학습 기법 (Focal Loss, Cosine Annealing, SWA)
def focal_loss(gamma=2.0, label_smoothing=0.05):
    def loss_fn(y_true, y_pred):
        num_cls   = tf.cast(tf.shape(y_true)[-1], tf.float32)
        y_true_sm = y_true * (1.0 - label_smoothing) + (label_smoothing / num_cls)
        y_pred    = tf.clip_by_value(y_pred, 1e-7, 1.0)
        ce        = -y_true_sm * tf.math.log(y_pred)
        weight    = y_true_sm * tf.math.pow(1.0 - y_pred, gamma)
        return tf.reduce_mean(tf.reduce_sum(weight * ce, axis=-1))
    return loss_fn

class CosineAnnealingWarmRestart(tf.keras.callbacks.Callback):
    def __init__(self, lr_max, lr_min, T0=10, T_mult=2):
        super().__init__()
        self.lr_max = lr_max; self.lr_min = lr_min
        self.T0 = T0; self.T_mult = T_mult; self.T_cur = 0; self.T_i = T0
    def on_epoch_begin(self, epoch, logs=None):
        lr = self.lr_min + 0.5 * (self.lr_max - self.lr_min) * (1 + math.cos(math.pi * self.T_cur / self.T_i))
        K.set_value(self.model.optimizer.learning_rate, lr)
        self.T_cur += 1
        if self.T_cur >= self.T_i:
            self.T_cur = 0; self.T_i = int(self.T_i * self.T_mult)

class SWACallback(tf.keras.callbacks.Callback):
    def __init__(self, start_epoch, swa_lr=1e-5):
        super().__init__()
        self.start_epoch = start_epoch; self.swa_lr = swa_lr
        self.swa_weights = None; self.n_averaged = 0
    def on_epoch_end(self, epoch, logs=None):
        if epoch >= self.start_epoch:
            if self.swa_weights is None:
                self.swa_weights = [w.copy() for w in self.model.get_weights()]
            else:
                for i, w in enumerate(self.model.get_weights()):
                    self.swa_weights[i] = (self.swa_weights[i] * self.n_averaged + w) / (self.n_averaged + 1)
            self.n_averaged += 1
            K.set_value(self.model.optimizer.learning_rate, self.swa_lr)
    def on_train_end(self, logs=None):
        if self.swa_weights is not None:
            self.model.set_weights(self.swa_weights)
            print(f"\n✅ SWA 완료: {self.n_averaged}개 평균")


## 🔟 모델 빌드 (ResNet50)

In [ ]:
# Cell 10: 모델 빌드 (ResNet50)
def build_model():
    base = ResNet50(weights="imagenet", include_top=False,
                    input_shape=(IMG_SIZE, IMG_SIZE, 3))
    base.trainable = False
    gap = GlobalAveragePooling2D()(base.output)
    gmp = GlobalMaxPooling2D()(base.output)
    x   = Concatenate()([gap, gmp])
    x = Dense(512, kernel_regularizer=l2(1e-4))(x)
    x = BatchNormalization()(x)
    x = tf.keras.layers.Activation("relu")(x)
    x = Dropout(0.35)(x)
    x = Dense(256, kernel_regularizer=l2(1e-4))(x)
    x = BatchNormalization()(x)
    x = tf.keras.layers.Activation("relu")(x)
    x = Dropout(0.25)(x)
    out = Dense(NUM_CLASSES, activation="softmax", dtype="float32")(x)
    return Model(inputs=base.input, outputs=out)

print("✅ ResNet50 모델 정의 완료")
print(f"   입력: ({IMG_SIZE}, {IMG_SIZE}, 3) | 출력: {NUM_CLASSES}클래스")
print(f"   헤드: GAP+GMP → Dense(512) → Dropout(0.35) → Dense(256) → Dropout(0.25) → Softmax")


## 🆕 10b. 과적합 / 과소적합 진단 함수

> 학습 히스토리(train loss vs val loss, train acc vs val acc)를 기록하고,
> 각 블록 크기·폴드별로 **과적합 / 과소적합 / 정상** 상태를 자동 판단합니다.


In [ ]:
# Cell 10b: 과적합/과소적합 진단 함수 & 모니터 콜백

def diagnose_fit(train_loss, val_loss, train_acc, val_acc):
    if not train_loss:
        return "unknown", {}
    final_tr_loss = train_loss[-1]
    final_va_loss = val_loss[-1]
    final_tr_acc  = train_acc[-1]
    final_va_acc  = val_acc[-1]
    gap_loss = final_va_loss - final_tr_loss
    gap_acc  = final_tr_acc  - final_va_acc

    if final_tr_acc < 0.60 and final_va_acc < 0.60:
        status = "underfit"
    elif gap_acc > 0.15 or gap_loss > 0.25:
        status = "overfit"
    else:
        status = "good"

    detail = {
        "train_loss_final": round(final_tr_loss, 4),
        "val_loss_final":   round(final_va_loss, 4),
        "train_acc_final":  round(final_tr_acc,  4),
        "val_acc_final":    round(final_va_acc,   4),
        "loss_gap":         round(gap_loss, 4),
        "acc_gap":          round(gap_acc,  4),
        "status":           status,
    }
    return status, detail


def print_fold_result(block_size, fold_idx, combined_history, stage=""):
    """폴드 학습 완료 시 결과를 텍스트로 출력"""
    tl = combined_history.get("loss", [])
    vl = combined_history.get("val_loss", [])
    ta = combined_history.get("accuracy", [])
    va = combined_history.get("val_accuracy", [])

    status, d = diagnose_fit(tl, vl, ta, va)

    STATUS_STR = {"good": "✅ 정상", "overfit": "🔴 과적합", "underfit": "🟡 과소적합", "unknown": "❓ 미상"}
    label = "원본" if block_size >= 512 else f"Block{block_size}"

    print()
    print(f"  ┌─────────────────────────────────────────────────────┐")
    print(f"  │  {label}  Fold {fold_idx+1} 학습 완료{stage:<24}│")
    print(f"  ├─────────────────────────────────────────────────────┤")
    print(f"  │  판정: {STATUS_STR[status]:<46}│")
    print(f"  │  train_acc : {d.get('train_acc_final',0)*100:5.1f}%   val_acc : {d.get('val_acc_final',0)*100:5.1f}%              │")
    print(f"  │  train_loss: {d.get('train_loss_final',0):6.4f}  val_loss: {d.get('val_loss_final',0):6.4f}               │")
    print(f"  │  acc_gap   : {d.get('acc_gap',0)*100:+5.1f}%  loss_gap: {d.get('loss_gap',0):+6.4f}               │")
    print(f"  │  총 에폭   : {len(tl):>3}  (1단계+2단계 합산){' '*17}│")

    # 간단 진단 코멘트
    ag = d.get('acc_gap', 0)
    lg = d.get('loss_gap', 0)
    if status == "overfit":
        if ag > 0.15:
            comment = f"  train-val acc 차이 {ag*100:.1f}%  →  Dropout/정규화 강화 권장"
        else:
            comment = f"  val_loss 급등 ({lg:+.4f})  →  EarlyStopping 작동 확인"
    elif status == "underfit":
        comment = "  train/val 모두 낮음  →  lr 상향 또는 에폭 추가 권장"
    else:
        comment = "  train ≈ val  →  정상 수렴"
    print(f"  │  💬 {comment:<50}│")
    print(f"  └─────────────────────────────────────────────────────┘")
    print()


# GPU 로그 저장소
GPU_LOG = []

class AdvancedMonitorCallback(Callback):
    """매 에폭마다 GPU 로깅 + 과적합 조기 경보"""
    def __init__(self, block_size, fold_idx, every_n_epochs=5, overfit_gap=0.20):
        super().__init__()
        self.block_size  = block_size
        self.fold_idx    = fold_idx
        self.every_n     = every_n_epochs
        self.overfit_gap = overfit_gap
        self._warned     = False

    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}

        # GPU 로깅
        if (epoch + 1) % self.every_n == 0:
            try:
                r = subprocess.run(
                    ["nvidia-smi",
                     "--query-gpu=memory.used,memory.total,utilization.gpu,temperature.gpu",
                     "--format=csv,noheader,nounits"],
                    capture_output=True, text=True, timeout=5
                )
                if r.returncode == 0:
                    parts = [p.strip() for p in r.stdout.strip().split(",")]
                    GPU_LOG.append({
                        "time":      time.time(),
                        "used_mib":  int(parts[0]),
                        "total_mib": int(parts[1]),
                        "util_pct":  float(parts[2]),
                        "temp_c":    float(parts[3]),
                        "epoch":     epoch + 1,
                        "block":     self.block_size,
                        "fold":      self.fold_idx,
                    })
                    used_pct = int(parts[0]) / int(parts[1]) * 100
                    if used_pct > 90:
                        print(f"\n  ⚠️  VRAM {used_pct:.1f}% — OOM 위험!")
            except:
                pass

        # 과적합 조기 경보
        tr_loss = logs.get("loss")
        va_loss = logs.get("val_loss")
        tr_acc  = logs.get("accuracy")
        va_acc  = logs.get("val_accuracy")

        if tr_loss is not None and va_loss is not None:
            loss_gap = va_loss - tr_loss
            acc_gap  = (tr_acc or 0) - (va_acc or 0)
            if loss_gap > self.overfit_gap and not self._warned:
                print(f"\n  🔴 [Epoch {epoch+1}] 과적합 경보! "
                      f"loss_gap={loss_gap:.4f}")
                self._warned = True
            elif acc_gap > 0.20 and not self._warned:
                print(f"\n  🔴 [Epoch {epoch+1}] 과적합 경보! "
                      f"acc_gap={acc_gap*100:.1f}%")
                self._warned = True


print("✅ diagnose_fit / print_fold_result / AdvancedMonitorCallback 정의 완료")


## 1️⃣1️⃣ 단일 블록 실험 함수

In [ ]:
# Cell 11: 단일 블록 크기 실험 함수 (히스토리 기록 + GPU 모니터 통합)
def run_experiment(block_size, block_dir):
    experiment_idx = BLOCK_SIZES.index(block_size) if block_size in BLOCK_SIZES else 0
    exp_seed = SEED + experiment_idx * 1000
    set_reproducible(exp_seed)
    label = "원본" if block_size >= IMG_SIZE else f"Block{block_size}"
    print(f"\n{'='*60}\n🧩 {label}\n{'='*60}")

    split_dir = (ORIGINAL_SPLIT_DIR if block_size >= IMG_SIZE else
                 os.path.join(WORKSPACE, f"AllExp_Split_Block{block_size}_{EXPERIMENT_VERSION}_seed{SEED}"))
    gc.collect()

    print("  오버샘플링...")
    oversampled_dir = oversample_dir(block_dir, TARGET_COUNT, seed=SEED)
    print("  Train/Test 분리...")
    prepare_split(oversampled_dir, split_dir, TEST_RATIO, seed=SEED)

    test_dir  = os.path.join(split_dir, "test")
    kfold_dir = os.path.join(split_dir, "kfold_data")

    class_names  = list_class_dirs(kfold_dir)
    class_to_idx = {c: i for i, c in enumerate(class_names)}
    benign_idx   = class_to_idx.get("Bengin cases", 0)
    normal_idx   = class_to_idx.get("Normal cases", 2)

    all_paths, all_labels = [], []
    for cls in class_names:
        for f in list_image_files(os.path.join(kfold_dir, cls)):
            all_paths.append(os.path.join(kfold_dir, cls, f))
            all_labels.append(class_to_idx[cls])
    all_paths  = np.array(all_paths)
    all_labels = np.array(all_labels)
    print(f"  학습: {len(all_paths)}장 | 테스트: "
          f"{sum(len(list_image_files(os.path.join(test_dir, c))) for c in class_names)}장")

    skf         = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
    fold_models = []

    # 블록 크기별 히스토리 초기화
    if block_size not in ALL_HISTORIES:
        ALL_HISTORIES[block_size] = {}

    for fold, (train_idx, val_idx) in enumerate(skf.split(all_paths, all_labels)):
        print(f"\n  --- Fold {fold+1}/{N_FOLDS} ---")
        gpu_status(f"Fold{fold+1} 시작")  # GPU 상태 출력
        set_reproducible(exp_seed + fold)

        fold_train_dir = os.path.join(split_dir, f"fold_{fold}_train")
        fold_val_dir   = os.path.join(split_dir, f"fold_{fold}_val")

        if not os.path.exists(fold_train_dir):
            for cls in class_names:
                os.makedirs(os.path.join(fold_train_dir, cls), exist_ok=True)
                os.makedirs(os.path.join(fold_val_dir,   cls), exist_ok=True)
            for idx in train_idx:
                src = all_paths[idx]; cls = class_names[all_labels[idx]]
                shutil.copy2(src, os.path.join(fold_train_dir, cls, os.path.basename(src)))
            for idx in val_idx:
                src = all_paths[idx]; cls = class_names[all_labels[idx]]
                shutil.copy2(src, os.path.join(fold_val_dir, cls, os.path.basename(src)))

        fold_cw = dict(enumerate(compute_class_weight(
            "balanced", classes=np.unique(all_labels[train_idx]), y=all_labels[train_idx])))
        fold_cw[benign_idx] = fold_cw[benign_idx] * 2.5
        fold_cw[normal_idx] = fold_cw[normal_idx] * 1.3

        train_ds, train_n = make_train_ds(fold_train_dir, class_names)
        val_ds, val_n, _  = make_eval_ds(fold_val_dir,   class_names)
        train_steps = math.ceil(train_n / BATCH_SIZE)
        val_steps   = math.ceil(val_n   / BATCH_SIZE)

        prefix = f"allexp_{block_size}_resnet50_f{fold}"
        TEMP   = f"{prefix}_temp.keras"
        BEST   = f"{prefix}_best.keras"

        monitor_cb = AdvancedMonitorCallback(block_size, fold, every_n_epochs=5)

        # ── 1단계: Feature extraction (백본 고정) ──────────────────
        model = build_model()
        model.compile(optimizer=AdamW(learning_rate=1e-3, weight_decay=1e-4),
                      loss=focal_loss(2.0, 0.02), metrics=["accuracy"])
        h1 = model.fit(
            apply_mixup_cutmix(train_ds, 0.3, cutmix_prob=0.5),
            steps_per_epoch=train_steps,
            validation_data=val_ds, validation_steps=val_steps,
            epochs=EPOCHS_S1,
            callbacks=[
                EarlyStopping(monitor="val_loss", patience=10,
                              restore_best_weights=True, verbose=0),
                ModelCheckpoint(TEMP, monitor="val_loss",
                                save_best_only=True, mode="min", verbose=0),
                CosineAnnealingWarmRestart(lr_max=1e-3, lr_min=1e-5, T0=5, T_mult=2),
                monitor_cb,
            ], class_weight=fold_cw, verbose=1)

        # ── 2단계: Fine-tuning (상위 30 레이어 해제) ───────────────
        model = load_model(TEMP, custom_objects={"loss_fn": focal_loss(2.0, 0.05)})
        bl = None
        for layer in model.layers:
            if hasattr(layer, "layers") and len(getattr(layer, "layers", [])) > 10:
                bl = layer; break
        if bl:
            bl.trainable = True
            for layer in bl.layers[:-30]: layer.trainable = False

        swa_cb = SWACallback(start_epoch=SWA_START, swa_lr=1e-5)
        model.compile(optimizer=AdamW(learning_rate=1e-4, weight_decay=1e-4),
                      loss=focal_loss(2.0, 0.02), metrics=["accuracy"])
        h2 = model.fit(
            apply_mixup_cutmix(train_ds, 0.1, cutmix_prob=0.3),
            steps_per_epoch=train_steps,
            validation_data=val_ds, validation_steps=val_steps,
            epochs=EPOCHS_S2,
            callbacks=[
                EarlyStopping(monitor="val_loss", patience=20,
                              restore_best_weights=False, verbose=0),
                ModelCheckpoint(BEST, monitor="val_loss",
                                save_best_only=True, mode="min", verbose=1),
                CosineAnnealingWarmRestart(lr_max=1e-4, lr_min=1e-7, T0=10, T_mult=2),
                swa_cb,
                monitor_cb,
            ], class_weight=fold_cw, verbose=1)

        # ── 히스토리 병합 저장 ────────────────────────────────────
        combined = {}
        for key in ["loss", "val_loss", "accuracy", "val_accuracy"]:
            combined[key] = list(h1.history.get(key, [])) + list(h2.history.get(key, []))
        ALL_HISTORIES[block_size][fold] = combined

        print_fold_result(block_size, fold, combined)

        model.save(BEST)
        fold_models.append(BEST)
        if os.path.exists(TEMP): os.remove(TEMP)
        del model
        tf.keras.backend.clear_session()
        gc.collect()
        gpu_status(f"Fold{fold+1} 완료")

    # ── 앙상블 + TTA ──────────────────────────────────────────────
    print(f"\n  🔮 앙상블 + TTA ({N_FOLDS}개 × {TTA_STEPS}회) [ResNet50]...")
    test_ds, test_n, test_files = make_eval_ds(test_dir, class_names)
    test_steps = math.ceil(test_n / BATCH_SIZE)

    y_true_list   = [yb.numpy() for _, yb in test_ds]
    y_true_onehot = np.concatenate(y_true_list, axis=0)
    y_true        = np.argmax(y_true_onehot, axis=1)

    all_fold_preds = []
    for fold, mp in enumerate(fold_models):
        fm = load_model(mp, custom_objects={"loss_fn": focal_loss(2.0, 0.05)})
        tta_preds = []
        for t in range(TTA_STEPS):
            set_reproducible(exp_seed + 10000 + fold * 100 + t)
            if t == 0:
                ds_t, _ = make_eval_ds(test_dir, class_names)[:2]
            else:
                ds_t, _ = make_tta_ds(test_dir, class_names)
            tta_preds.append(fm.predict(ds_t, steps=test_steps, verbose=0))
            print(f"    Fold{fold+1} TTA {t+1}/{TTA_STEPS}", end="\r")
        all_fold_preds.append(np.mean(tta_preds, axis=0))
        del fm; tf.keras.backend.clear_session(); gc.collect()
    print()

    pred_probs = np.mean(all_fold_preds, axis=0)[:len(y_true)]
    y_pred     = np.argmax(pred_probs, axis=1)
    test_acc   = float(np.mean(y_true == y_pred))
    print(f"  앙상블: {test_acc*100:.2f}%")

    # ── Pseudo Labeling ───────────────────────────────────────────
    print("  🔄 Pseudo Labeling...")
    pseudo_dir = os.path.join(split_dir, "pseudo")
    if os.path.exists(pseudo_dir): shutil.rmtree(pseudo_dir)
    pseudo_count = 0
    for fpath, prob in zip(test_files, pred_probs):
        if np.max(prob) >= 0.88:
            cls = class_names[np.argmax(prob)]
            os.makedirs(os.path.join(pseudo_dir, cls), exist_ok=True)
            shutil.copy2(fpath, os.path.join(pseudo_dir, cls, os.path.basename(fpath)))
            pseudo_count += 1
    print(f"  확신도 88%↑: {pseudo_count}장")

    if pseudo_count >= 10:
        pseudo_kfold = os.path.join(split_dir, "pseudo_kfold")
        if os.path.exists(pseudo_kfold): shutil.rmtree(pseudo_kfold)
        for cls in class_names:
            os.makedirs(os.path.join(pseudo_kfold, cls), exist_ok=True)
            for f in list_image_files(os.path.join(kfold_dir, cls)):
                shutil.copy2(os.path.join(kfold_dir, cls, f),
                             os.path.join(pseudo_kfold, cls, f))
            p = os.path.join(pseudo_dir, cls)
            if os.path.exists(p):
                for f in list_image_files(p):
                    shutil.copy2(os.path.join(p, f),
                                 os.path.join(pseudo_kfold, cls, f"pseudo_{f}"))

        set_reproducible(exp_seed + 50000)
        pm = load_model(fold_models[0], custom_objects={"loss_fn": focal_loss(2.0, 0.05)})
        p_labels = []
        for cls in class_names:
            cnt = len(list_image_files(os.path.join(pseudo_kfold, cls)))
            p_labels.extend([class_to_idx[cls]] * cnt)
        p_cw = dict(enumerate(compute_class_weight(
            "balanced", classes=np.unique(p_labels), y=p_labels)))
        p_cw[benign_idx] = p_cw[benign_idx] * 1.5
        p_cw[normal_idx] = p_cw[normal_idx] * 1.3

        p_train_ds, p_n     = make_train_ds(pseudo_kfold, class_names)
        p_val_ds, p_vn, _   = make_eval_ds(os.path.join(split_dir, "fold_0_val"), class_names)
        pm.compile(optimizer=AdamW(learning_rate=3e-5, weight_decay=1e-4),
                   loss=focal_loss(2.0, 0.05), metrics=["accuracy"])
        pm.fit(apply_mixup(p_train_ds, 0.1),
               steps_per_epoch=math.ceil(p_n / BATCH_SIZE),
               validation_data=p_val_ds, validation_steps=math.ceil(p_vn / BATCH_SIZE),
               epochs=15,
               callbacks=[EarlyStopping(monitor="val_loss", patience=7,
                                        restore_best_weights=True, verbose=0)],
               class_weight=p_cw, verbose=0)

        tta_preds = []
        for t in range(TTA_STEPS):
            set_reproducible(exp_seed + 30000 + t)
            if t == 0:
                ds_t, _ = make_eval_ds(test_dir, class_names)[:2]
            else:
                ds_t, _ = make_tta_ds(test_dir, class_names)
            tta_preds.append(pm.predict(ds_t, steps=test_steps, verbose=0))
        all_fold_preds.append(np.mean(tta_preds, axis=0))
        del pm; tf.keras.backend.clear_session(); gc.collect()

        pred_probs = np.mean(all_fold_preds, axis=0)[:len(y_true)]
        y_pred     = np.argmax(pred_probs, axis=1)
        test_acc   = float(np.mean(y_true == y_pred))
        print(f"  Pseudo 후: {test_acc*100:.2f}%")

    cm      = confusion_matrix(y_true, y_pred)
    roc_auc = float(roc_auc_score(y_true_onehot[:len(pred_probs)], pred_probs,
                                   multi_class="ovr", average="macro"))
    per_class = {cls: float(cm[i, i] / cm[i].sum()) for i, cls in enumerate(class_names)}

    for mp in fold_models:
        if os.path.exists(mp): os.remove(mp)


    print(f"\n  ✅ Block{block_size}: {test_acc*100:.2f}% | ROC-AUC={roc_auc:.4f}")
    return {"block_size": block_size,
            "experiment_seed": exp_seed,
            "acc": test_acc, "roc_auc": roc_auc,
            "per_class_acc": per_class, "cm": cm.tolist(), "class_names": class_names}


## 1️⃣2️⃣ 전체 실험 루프

In [ ]:
# Cell 12: 전체 실험 루프 (순차 실행)
print("\n" + "=" * 60)
print("🔬 블록 암호화 크기별 실험 시작 (ResNet50)")
print(f"   블록: {BLOCK_SIZES}")
print(f"   K={N_FOLDS} | TTA={TTA_STEPS} | BATCH={BATCH_SIZE} | IMG={IMG_SIZE}")
print(f"   클래스별 오버샘플링: {TARGET_COUNT}장")
print(f"   실험 버전: {EXPERIMENT_VERSION}")
print("=" * 60)
gpu_status("실험 시작 전")

all_results = {}
result_path = os.path.join(WORKSPACE, f"all_block_results_resnet50_{EXPERIMENT_VERSION}.json")

if os.path.exists(result_path):
    with open(result_path) as f:
        saved = json_module.load(f)
    for k, v in saved.items():
        all_results[int(k)] = v
    print(f"\n✅ 이전 결과: {sorted(all_results.keys())}")

for step_idx, block_size in enumerate(BLOCK_SIZES, start=1):
    if block_size in all_results:
        print(f"\n⏭️  Step {step_idx}/{len(BLOCK_SIZES)} | Block{block_size}: 완료({all_results[block_size]['acc']*100:.2f}%) → 스킵")
        continue
    print(f"\n📦 Step {step_idx}/{len(BLOCK_SIZES)} | Block{block_size} 데이터셋 준비...")
    block_dir = generate_block_dataset(block_size)
    result    = run_experiment(block_size, block_dir)
    all_results[block_size] = result
    with open(result_path, "w") as f:
        json_module.dump({str(k): v for k, v in all_results.items()}, f, indent=2, ensure_ascii=False)
    print(f"  💾 중간 저장 완료")
    gpu_status(f"Block{block_size} 완료 후")


## 1️⃣3️⃣ 결과 표 출력

In [ ]:
# Cell 13: 결과 표 출력
print("\n\n" + "=" * 85)
print("📊 블록 암호화 크기별 실험 결과 (ResNet50)")
print("=" * 85)
print(f"  {'블록크기':>8}  {'정확도(전체)':>12}  {'ROC-AUC':>8}  {'양성(Benign)':>13}  {'악성(Malig)':>12}  {'정상(Normal)':>13}  {'평가':>6}")
print("-" * 85)
for bs in BLOCK_SIZES:
    if bs not in all_results: continue
    r   = all_results[bs]
    b   = r["per_class_acc"].get("Bengin cases",    0) * 100
    m   = r["per_class_acc"].get("Malignant cases", 0) * 100
    n   = r["per_class_acc"].get("Normal cases",    0) * 100
    acc = r["acc"] * 100
    lbl = "원본" if bs >= 512 else f"{bs}x{bs}"
    grade = "🟢 우수" if acc >= 85 else ("🟡 양호" if acc >= 70 else ("🟠 미흡" if acc >= 55 else "🔴 불량"))
    print(f"  {lbl:>8}  {acc:>11.2f}%  {r['roc_auc']:>8.4f}"
          f"  {b:>12.1f}%  {m:>11.1f}%  {n:>12.1f}%  {grade}")
print("=" * 85)


## 🆕 13b. 전체 과적합 / 과소적합 진단 & GPU 타임라인

> 모든 블록 크기·폴드에 걸쳐 과적합 상태를 한눈에 보여줍니다.


In [ ]:
# Cell 13b: 전체 과적합/과소적합 진단 요약
print("=" * 75)
print("🩺 전체 과적합 / 과소적합 진단 요약 [ResNet50 블록암호화]")
print("=" * 75)
STATUS_EMOJI = {"good": "✅ 정상", "overfit": "🔴 과적합", "underfit": "🟡 과소적합", "unknown": "❓ 미상"}

for bs in BLOCK_SIZES:
    if bs not in ALL_HISTORIES:
        continue
    lbl = "원본(512)" if bs >= 512 else (f"픽셀 1×1" if bs == 1 else f"Block{bs}×{bs}")
    print(f"\n  [{lbl}]")
    statuses = []
    for fold_idx, hist in sorted(ALL_HISTORIES[bs].items()):
        status, detail = diagnose_fit(
            hist.get("loss", []), hist.get("val_loss", []),
            hist.get("accuracy", []), hist.get("val_accuracy", [])
        )
        statuses.append(status)
        print(f"    Fold {fold_idx+1}: {STATUS_EMOJI[status]:<12}  "
              f"train={detail.get('train_acc_final',0)*100:.1f}%  "
              f"val={detail.get('val_acc_final',0)*100:.1f}%  "
              f"acc_gap={detail.get('acc_gap',0)*100:+.1f}%  "
              f"loss_gap={detail.get('loss_gap',0):+.4f}")
    cnt = {s: statuses.count(s) for s in ["good", "overfit", "underfit"]}
    dominant = max(cnt, key=cnt.get)
    print(f"    → 종합: {STATUS_EMOJI[dominant]}  "
          f"(정상 {cnt['good']}폴드 / 과적합 {cnt['overfit']}폴드 / 과소적합 {cnt['underfit']}폴드)")

print("=" * 75)
